In [4]:
import pandas as pd
import os

In [98]:
path = os.getcwd()
path = os.path.split(path)[0]
path = os.path.split(path)[0]
path = os.path.join(path, r"data/Combined_Residential_Sold.csv")


In [99]:
df = pd.read_csv(path)
original_combined_data_cols = df.columns

C:\Users\fistb\AppData\Local\Temp\ipykernel_9076\2807247665.py:1: DtypeWarning: Columns (46: BuyerAgencyCompensationType, 65: FireplaceYN, 79: latfilled, 80: lonfilled, 81: BuyerAgentAOR, 82: ListAgentAOR, 83: OriginatingSystemName, 84: OriginatingSystemSubName) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [21]:
path = os.getcwd()
path = os.path.split(path)[0]
path = os.path.split(path)[0]
path = os.path.join(path, r"data/Filtered_sold_data.csv")


In [22]:
df = pd.read_csv(path)
week2_3_data_cols = df.columns

C:\Users\fistb\AppData\Local\Temp\ipykernel_9076\950691558.py:1: DtypeWarning: Columns (41: BuyerAgencyCompensationType, 65: latfilled, 66: lonfilled, 67: BuyerAgentAOR, 68: ListAgentAOR, 69: OriginatingSystemName, 70: OriginatingSystemSubName) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [23]:
print(f"Original Data had {len(original_combined_data_cols)}")
print(f'After Week 3, data has {len(week2_3_data_cols)} Columns. {len(original_combined_data_cols) - len(week2_3_data_cols)} were dropped in Week 3!')

Original Data had 85
After Week 3, data has 71 Columns. 14 were dropped in Week 3!


In [26]:
datetime_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
df[datetime_cols].head()

,CloseDate,PurchaseContractDate,ListingContractDate,ContractStatusChangeDate
0,2024-01-02,2023-11-15,2023-11-15,2024-01-02
1,2024-01-30,2024-01-13,2024-01-12,2024-01-30
2,2024-01-16,2024-01-10,2023-12-07,2024-01-16
3,2024-01-24,2023-12-08,2023-12-07,2024-01-24
4,2024-01-31,2023-12-23,2023-12-21,2024-01-31


In [28]:
df[datetime_cols] = df[datetime_cols].apply(pd.to_datetime)

In [84]:
original_row_count = df.shape[0]
original_row_count 

354197

In [42]:
na_pt = (df.isna().sum()/df.shape[0]).sort_values(ascending=False) 

In [47]:
cols_to_drop = na_pt[na_pt > 0.7].index.to_list() 
cols_to_drop

['OriginatingSystemSubName',
 'OriginatingSystemName',
 'ElementarySchool',
 'MiddleOrJuniorSchool',
 'BuyerAgencyCompensation',
 'BuyerAgencyCompensationType',
 'HighSchool',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'CoListOfficeName',
 'latfilled',
 'lonfilled']

In [49]:
df = df.drop(columns=cols_to_drop)

In [51]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'AttachedGarageYN',
       'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName',
       'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionY

# Timeline Checking and Flagging

In [78]:
df['listing_after_close_flag'] = df['CloseDate'] < df['ListingContractDate']
df['purchase_after_close_flag'] = df['CloseDate'] < df['PurchaseContractDate']
df['negative_flag'] = df['DaysOnMarket'] < 0


# Geographic checking and flagging

In [79]:
df['missing_coords'] = df['Latitude'].isna() | df['Longitude'].isna()

df['invalid_lat_or_long'] = (df['Latitude'] == 0) | (df['Longitude'] == 0)

df['Long_greater_0'] = df['Longitude'] > 0

df['out_of_state_flag'] = ~(
    df['Longitude'].between(-125, -110) &
    df['Latitude'].between(30, 45)
)

In [80]:
flag_cols = [
    'listing_after_close_flag',
    'purchase_after_close_flag',
    'negative_flag',
    'missing_coords',
    'invalid_lat_or_long',
    'Long_greater_0',
    'out_of_state_flag',
]

In [81]:
test = [False, False, False, False, False, False, True]

In [82]:
for col in flag_cols:
    print(df[col].value_counts())
    print()

listing_after_close_flag
False    354157
True         40
Name: count, dtype: int64

purchase_after_close_flag
False    354165
True         32
Name: count, dtype: int64

negative_flag
False    354197
Name: count, dtype: int64

missing_coords
False    351189
True       3008
Name: count, dtype: int64

invalid_lat_or_long
False    354174
True         23
Name: count, dtype: int64

Long_greater_0
False    354171
True         26
Name: count, dtype: int64

out_of_state_flag
False    351128
True       3069
Name: count, dtype: int64



In [83]:
df['Should_drop'] = False
for col in flag_cols:
    df['Should_drop'] = df['Should_drop']  | df[col] 


In [85]:
df['Should_drop'].value_counts()

Should_drop
False    351059
True       3138
Name: count, dtype: int64

In [86]:
cleaned_df = df[~df['Should_drop']].copy()

In [89]:
rows_dropped = original_row_count - cleaned_df.shape[0]

In [93]:
cleaned_df.to_csv("Cleaned_sold.csv")

# Conclusion

In [100]:
print(f"The Original Data had {len(original_combined_data_cols)} and {original_row_count}!")

The Original Data had 85 and 354197!


In [105]:
print(f'After processing, the data now has only {len(cleaned_df.columns)} cols and {cleaned_df.shape[0]} rows!')

After processing, the data now has only 67 cols and 351059 rows!


In [106]:
print(f'In total, {len(original_combined_data_cols) - len(cleaned_df.columns)} Columns and {rows_dropped} Rows were dropped from Sold Dataset!')

In total, 18 Columns and 3138 Rows were dropped from Sold Dataset!
